<a href="https://colab.research.google.com/github/kimnguyen02/ITAI-2372-_-Module-3_-AI-in-Healthcare/blob/main/AI_Powered_Symptom_Checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Symptom Checker
## Module 3 - ITAI 2372
### Student: Kim Nguyen

CREATE A SMALL DATASET symptom_data.csv


In [1]:
import pandas as pd

# Create a small educational symptom dataset
data = {
    "category": [
        "Flu-like illness",
        "Cold-like illness",
        "Migraine-like symptoms",
        "Gastrointestinal illness",
        "Respiratory symptoms",
        "Allergic reaction",
        "Emergency warning signs"
    ],

    "symptoms": [
        "fever cough sore throat fatigue headache",
        "cough runny nose sneezing sore throat congestion",
        "headache nausea sensitivity light migraine",
        "nausea vomiting diarrhea stomach pain",
        "cough wheezing breathing difficulty chest tightness",
        "sneezing runny nose itchy eyes congestion",
        "severe chest pain severe breathing difficulty fainting confusion"
    ],

    "triage": [
        "Routine",
        "Self-care",
        "Routine",
        "Routine",
        "Urgent",
        "Self-care",
        "Emergency"
    ]
}

# Convert the dictionary into a DataFrame
df = pd.DataFrame(data)

# Save the dataset as a CSV file
df.to_csv("symptom_data.csv", index=False)

# Display the dataset
df

,category,symptoms,triage
0,Flu-like illness,fever cough sore throat fatigue headache,Routine
1,Cold-like illness,cough runny nose sneezing sore throat congestion,Self-care
2,Migraine-like symptoms,headache nausea sensitivity light migraine,Routine
3,Gastrointestinal illness,nausea vomiting diarrhea stomach pain,Routine
4,Respiratory symptoms,cough wheezing breathing difficulty chest tigh...,Urgent
5,Allergic reaction,sneezing runny nose itchy eyes congestion,Self-care
6,Emergency warning signs,severe chest pain severe breathing difficulty ...,Emergency


In [2]:
# Read the CSV file back into Python
test_df = pd.read_csv("symptom_data.csv")

# Display the first few rows
test_df.head()

,category,symptoms,triage
0,Flu-like illness,fever cough sore throat fatigue headache,Routine
1,Cold-like illness,cough runny nose sneezing sore throat congestion,Self-care
2,Migraine-like symptoms,headache nausea sensitivity light migraine,Routine
3,Gastrointestinal illness,nausea vomiting diarrhea stomach pain,Routine
4,Respiratory symptoms,cough wheezing breathing difficulty chest tigh...,Urgent


In [3]:
# Import libraries needed
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# Load symptom dataset
df = pd.read_csv("symptom_data.csv")

# Display the dataset
df

,category,symptoms,triage
0,Flu-like illness,fever cough sore throat fatigue headache,Routine
1,Cold-like illness,cough runny nose sneezing sore throat congestion,Self-care
2,Migraine-like symptoms,headache nausea sensitivity light migraine,Routine
3,Gastrointestinal illness,nausea vomiting diarrhea stomach pain,Routine
4,Respiratory symptoms,cough wheezing breathing difficulty chest tigh...,Urgent
5,Allergic reaction,sneezing runny nose itchy eyes congestion,Self-care
6,Emergency warning signs,severe chest pain severe breathing difficulty ...,Emergency


In [5]:
# Create a TF-IDF vectorizer
# TF-IDF converts text into numerical values that the computer can compare.
vectorizer = TfidfVectorizer()

# Learn the important words from our symptom descriptions
symptom_vectors = vectorizer.fit_transform(df["symptoms"])

print("TF-IDF matrix shape:", symptom_vectors.shape)

TF-IDF matrix shape: (7, 28)


CREATE A SYMPTOM ANALYSIS FUNCTION

In [6]:
def analyze_symptoms(user_input):
    """
    Analyze a user's symptom description.

    The function:
    1. Converts the user's text into TF-IDF values.
    2. Compares it with each symptom category.
    3. Finds the category with the highest similarity.
    4. Returns the category and triage level.
    """

    # Convert the user's symptom description into TF-IDF values
    user_vector = vectorizer.transform([user_input])

    # Calculate similarity between the user's symptoms
    # and each symptom category in the dataset
    similarities = cosine_similarity(user_vector, symptom_vectors)[0]

    # Find the index of the highest similarity score
    best_match_index = similarities.argmax()

    # Get information about the best matching category
    best_category = df.iloc[best_match_index]["category"]
    triage_level = df.iloc[best_match_index]["triage"]

    # Get the similarity score
    similarity_score = similarities[best_match_index]

    return best_category, triage_level, similarity_score

Testing

In [7]:
user_symptoms = "I have a fever, headache, cough and sore throat."

category, triage, score = analyze_symptoms(user_symptoms)

print("User symptoms:", user_symptoms)
print("Possible category:", category)
print("Triage level:", triage)
print("Similarity score:", round(score, 3))

User symptoms: I have a fever, headache, cough and sore throat.
Possible category: Flu-like illness
Triage level: Routine
Similarity score: 0.884


Add emergency keyword check and a minimum similarity threshold

In [8]:
# Emergency symptoms that should trigger an immediate warning.
emergency_terms = [
    "severe chest pain",
    "chest pain",
    "severe breathing difficulty",
    "difficulty breathing",
    "cannot breathe",
    "fainting",
    "unconscious",
    "severe confusion"
]

def analyze_symptoms(user_input):
    """
    Analyze a user's symptom description.

    The function:
    1. Checks for simplified emergency warning terms.
    2. Converts the user's text into TF-IDF values.
    3. Calculates cosine similarity.
    4. Finds the closest symptom category.
    5. Uses a similarity threshold to avoid unreliable matches.
    """

    # Convert the user's input to lowercase
    text = user_input.lower()

    # Check for emergency warning terms first
    for term in emergency_terms:
        if term in text:
            return (
                "Emergency warning signs",
                "Emergency",
                1.0
            )

    # Convert the user's symptoms into TF-IDF values
    user_vector = vectorizer.transform([user_input])

    # Compare the user's symptoms with the dataset
    similarities = cosine_similarity(
        user_vector,
        symptom_vectors
    )[0]

    # Find the category with the highest similarity
    best_match_index = similarities.argmax()

    # Get the highest similarity score
    similarity_score = similarities[best_match_index]

    # Set a minimum similarity threshold
    minimum_similarity = 0.20

    # If the similarity is too low, don't make a classification
    if similarity_score < minimum_similarity:
        return (
            "No reliable match",
            "Uncertain",
            similarity_score
        )

    # Get the matching category and triage level
    best_category = df.iloc[best_match_index]["category"]
    triage_level = df.iloc[best_match_index]["triage"]

    return (
        best_category,
        triage_level,
        similarity_score
    )

Testing

In [9]:
test_inputs = [
    "I have a fever, cough and sore throat.",
    "I have a headache and nausea.",
    "I have a runny nose and itchy eyes.",
    "I have severe chest pain and difficulty breathing.",
    "My finger hurts after an accident."
]

for symptoms in test_inputs:
    category, triage, score = analyze_symptoms(symptoms)

    print("-----------------------------------")
    print("Symptoms:", symptoms)
    print("Category:", category)
    print("Triage:", triage)
    print("Similarity:", round(score, 3))

-----------------------------------
Symptoms: I have a fever, cough and sore throat.
Category: Flu-like illness
Triage: Routine
Similarity: 0.794
-----------------------------------
Symptoms: I have a headache and nausea.
Category: Migraine-like symptoms
Triage: Routine
Similarity: 0.561
-----------------------------------
Symptoms: I have a runny nose and itchy eyes.
Category: Allergic reaction
Triage: Self-care
Similarity: 0.843
-----------------------------------
Symptoms: I have severe chest pain and difficulty breathing.
Category: Emergency warning signs
Triage: Emergency
Similarity: 1.0
-----------------------------------
Symptoms: My finger hurts after an accident.
Category: No reliable match
Triage: Uncertain
Similarity: 0.0


## Disclaimer
Important: This application is an educational AI prototype and is not a medical diagnostic tool. It does not provide medical advice or replace evaluation by a qualified healthcare professional. If you believe you may be experiencing a medical emergency, seek immediate professional medical care.

CREATE A STREAMLIT INTERFACE

In [10]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 31.0 MB/s eta 0:00:00


In [11]:
# Create app.py
%%writefile app.py

import pandas as pd
import streamlit as st

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. LOAD THE DATASET

# Load the symptom dataset.
df = pd.read_csv("symptom_data.csv")

# 2. CREATE THE TF-IDF MODEL

# TF-IDF converts text into numerical values.
vectorizer = TfidfVectorizer()

# Learn the important words from the symptom descriptions.
symptom_vectors = vectorizer.fit_transform(df["symptoms"])


# 3. EMERGENCY WARNING TERMS

emergency_terms = [
    "severe chest pain",
    "chest pain",
    "severe breathing difficulty",
    "difficulty breathing",
    "cannot breathe",
    "fainting",
    "unconscious",
    "severe confusion"
]

# 4. SYMPTOM ANALYSIS FUNCTION

def analyze_symptoms(user_input):
    """
    Analyze the user's symptom description.

    The function:
    1. Checks for simplified emergency warning terms.
    2. Converts the input into TF-IDF values.
    3. Calculates cosine similarity.
    4. Finds the closest symptom category.
    5. Uses a similarity threshold for uncertain cases.
    """

    # Convert the user's input to lowercase.
    text = user_input.lower()

    # Check for emergency warning terms first.
    for term in emergency_terms:
        if term in text:
            return (
                "Emergency warning signs",
                "Emergency",
                1.0
            )

    # Convert the user's symptoms into TF-IDF values.
    user_vector = vectorizer.transform([user_input])

    # Compare the user's symptoms with the dataset.
    similarities = cosine_similarity(
        user_vector,
        symptom_vectors
    )[0]

    # Find the category with the highest similarity.
    best_match_index = similarities.argmax()

    # Get the similarity score.
    similarity_score = similarities[best_match_index]

    # Minimum similarity required for a classification.
    minimum_similarity = 0.20

    # If the similarity is too low, return an uncertain result.
    if similarity_score < minimum_similarity:
        return (
            "No reliable match",
            "Uncertain",
            similarity_score
        )

    # Get the matching category.
    best_category = df.iloc[best_match_index]["category"]

    # Get the associated triage level.
    triage_level = df.iloc[best_match_index]["triage"]

    return (
        best_category,
        triage_level,
        similarity_score
    )


# 5. STREAMLIT USER INTERFACE

st.title("AI-Powered Symptom Checker")

st.write(
    "This educational application uses basic natural language "
    "processing to compare a user's symptom description with "
    "a small symptom dataset."
)


# Healthcare disclaimer
st.warning(
    "Important: This application is an educational AI prototype "
    "and is NOT a medical diagnostic tool. It does not provide "
    "medical advice or replace evaluation by a qualified healthcare "
    "professional. If you believe you are experiencing a medical "
    "emergency, seek immediate professional medical care."
)


# Text box for the user's symptoms
user_input = st.text_area(
    "Describe your symptoms:",
    placeholder="Example: I have a fever, cough and sore throat."
)


# Analyze button
if st.button("Analyze Symptoms"):

    # Make sure the user entered something.
    if not user_input.strip():

        st.error("Please enter your symptoms first.")

    else:

        # Analyze the user's symptoms.
        category, triage, score = analyze_symptoms(user_input)

        st.subheader("Result")

        st.write("**Possible symptom category:**", category)

        st.write("**Educational triage level:**", triage)

        st.write(
            "**Similarity score:**",
            round(score, 3)
        )

        # Additional warning for uncertain results.
        if triage == "Uncertain":

            st.info(
                "The application could not find a reliable match. "
                "Please do not use this result to make medical decisions."
            )

        # Emergency warning.
        elif triage == "Emergency":

            st.error(
                "Emergency warning signs were detected. "
                "Seek immediate professional medical attention."
            )

Writing app.py


In [12]:
!ls

app.py	sample_data  symptom_data.csv


In [13]:
%%writefile requirements.txt

pandas
scikit-learn
streamlit

Writing requirements.txt


In [14]:
%%writefile README.md

# AI-Powered Symptom Checker and Triage Tool

## Project Overview

This project is a simple AI-driven healthcare application developed
for educational purposes.

The application uses Natural Language Processing (NLP) to analyze
user-entered symptom descriptions and compare them with a small
educational symptom dataset.

The system uses TF-IDF vectorization and cosine similarity to identify
the most similar symptom category and provide an educational triage
level.

## How It Works

The application follows these steps:

1. The user enters a description of their symptoms.
2. The application processes the text.
3. TF-IDF converts the symptom descriptions into numerical features.
4. Cosine similarity compares the user's input with the symptom dataset.
5. The system identifies the closest symptom category.
6. The application displays an educational triage level.
7. If the similarity is too low, the system returns an uncertain result.
8. Simplified emergency warning terms trigger an emergency warning.

## Technologies Used

- Python
- Pandas
- Scikit-learn
- Streamlit
- TF-IDF
- Cosine Similarity
- Natural Language Processing

## Dataset

The project uses a small, manually created educational dataset
containing symptom descriptions, categories, and triage levels.

This dataset is not intended to represent a clinical or medically
validated dataset.

## Example

Example input:

"I have a fever, cough and sore throat."

The application may return:

- Possible category: Flu-like illness
- Triage level: Routine

## Testing

The application was tested using several different symptom descriptions,
including:

- Fever, cough and sore throat
- Headache and nausea
- Runny nose and itchy eyes
- Severe chest pain and difficulty breathing
- An unrelated finger injury

The system successfully identified matching categories and returned an
uncertain result when the input did not sufficiently match the dataset.

## Ethical Considerations

### Medical Safety

This application is an educational prototype and is not a medical
diagnostic system.

It should not be used to diagnose or treat medical conditions.

### Data Quality

The application's performance depends heavily on the quality and
coverage of the symptom dataset.

### Bias

A small dataset may not represent the full diversity of real-world
healthcare situations and may produce unreliable results for symptoms
not represented in the dataset.

### Privacy

The prototype does not store or transmit user-entered symptoms.

A real healthcare application would require appropriate privacy,
security, and regulatory safeguards.

## Limitations

- Small educational dataset
- Limited symptom categories
- No clinical validation
- Simple NLP approach
- No patient medical history
- No demographic or clinical context
- Not suitable for real medical decision-making

## Disclaimer

This application is for educational purposes only and is not a
substitute for professional medical advice, diagnosis, or treatment.

If you believe you are experiencing a medical emergency, seek
immediate professional medical care.

## Future Improvements

Future versions could include:

- A larger medically validated dataset
- More advanced NLP models
- Better synonym and phrase recognition
- Clinical validation
- Improved uncertainty detection
- More comprehensive safety rules
- Secure handling of patient data

Writing README.md
